##### ① 구조에서 서열 뽑기, FASTA 저장

In [3]:
import os
os.chdir("/home/kjk/work") 

In [4]:
import re
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1    # 3글자 → 1글자 변환: seq1("TRP") == "W"

model = PDBParser(QUIET=True).get_structure("c", "data/raw/5X8M.pdb")[0]

def chain_sequence(chain):
    residues = [r for r in chain if r.id[0] == " "]             # 잔기 정보 추출(리스트 컴프리헨션)
    seq = "".join(seq1(r.get_resname()) for r in residues)      # 1-letter 잔기서열 문자열로
    return seq, residues


# chain ID: (서열, 잔기 리스트)
chains = {chain.id: chain_sequence(chain) for chain in model}

for chain_id, (seq, _) in chains.items():    # _ : 쓰지 않는 값을 받는 관례
    print(chain_id, len(seq), seq[:30] + "...")

# FASTA 형식: '>이름' 줄 다음에 서열 (보통 60자마다 줄바꿈)
with open("results/5X8M_observed.fasta", "w") as f:
    for chain_id, (seq, _) in chains.items():
        f.write(f">5X8M_{chain_id} observed residues\n")
        for k in range(0, len(seq), 60):     # 0, 60, 120 … 간격으로
            f.write(seq[k:k + 60] + "\n")


A 116 AFTVTVPKDLYVVEYGSNMTIECKFPVEKE...
B 214 EVQLVESGGGLVQPGGSLRLSCAASGFTFS...
C 213 IVLTQSPGTLSLSPGERATLSCRASQRVSS...


##### ② CDR-H3, L3 찾기

In [5]:
# 모티프 패턴. 괄호 안이 CDR
H3_PATTERN = re.compile(r"C[A-Z]{2}([A-Z]{3,25}?)WG[A-Z]G")
L3_PATTERN = re.compile(r"C([A-Z]{7,11}?)FG[A-Z]G")


def find_cdr(seq, residues, pattern):
    # 서열에서 패턴을 찾아 (CDR 서열, 시작 잔기 번호, 끝 잔기 번호)를 돌려준다. 못 찾으면 None
    m = pattern.search(seq)
    if m is None:
        return None
    start, end = m.start(1), m.end(1)          # 서열 인덱스 (end는 미포함)
    # 서열 인덱스 → PDB 잔기 번호. Day 1에서 봤듯이 둘은 다를 수 있다!
    return m.group(1), residues[start].id[1], residues[end - 1].id[1]


for chain_id, pattern, name in (("B", H3_PATTERN, "CDR-H3"), ("C", L3_PATTERN, "CDR-L3")):
    seq, residues = chains[chain_id]
    print(name, find_cdr(seq, residues, pattern))


CDR-H3 ('EGGWFGELAFDY', 99, 110)
CDR-L3 ('QQYGSLPWT', 90, 98)


##### ③ paratope을 CDR에 대응

In [6]:
CDR_RANGES = {
    "B": {"H1": (31, 35), "H2": (50, 66), "H3": (99, 110)},
    "C": {"L1": (24, 35), "L2": (51, 57), "L3": (90, 98)},
}

PARATOPE = {
    "B": [33, 50, 52, 57, 59, 102, 103],
    "C": [28, 30, 31, 33, 92, 93, 94, 95, 97],
}


def which_cdr(chain_id, resnum):
    # 잔기 번호가 속한 CDR 이름을 돌려주고, 어디에도 없으면 'FR'을 돌려준다
    for name, (start, end) in CDR_RANGES[chain_id].items():
        if start <= resnum <= end:         # 연쇄 비교
            return name
    return "FR"


for chain_id, nums in PARATOPE.items():
    for num in nums:
        print(chain_id, num, which_cdr(chain_id, num))


B 33 H1
B 50 H2
B 52 H2
B 57 H2
B 59 H2
B 102 H3
B 103 H3
C 28 L1
C 30 L1
C 31 L1
C 33 L1
C 92 L3
C 93 L3
C 94 L3
C 95 L3
C 97 L3
